# NLP Project — Insurer Reviews Analysis
Column used: `avis_en` | Translation: skipped | GPU: RTX 6000 Ada 48GB

---
## Cell 1 — Global imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings, os, re, json, pickle
from collections import Counter
from tqdm.auto import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.decomposition import LatentDirichletAllocation

import gensim
from gensim.models import Word2Vec
import gensim.downloader as api

import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
    pipeline, BitsAndBytesConfig
)
from datasets import Dataset
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from sentence_transformers import SentenceTransformer

import umap
from scipy.spatial.distance import cosine, euclidean

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout,
    Conv1D, GlobalMaxPooling1D, Bidirectional
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import TensorBoard as TFTensorBoard

nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt_tab', quiet=True)

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

os.makedirs('./checkpoints', exist_ok=True)
print('Imports done.')

---
## Cell 2 — Load data

In [ ]:
df = pd.read_csv('reviews_concat_prepared.csv')
df['avis_en']    = df['avis_en'].fillna('')
df['avis_clean'] = df['avis_clean'].fillna('') if 'avis_clean' in df.columns else df['avis_en']

print(f'Shape      : {df.shape}')
print(f'Columns    : {df.columns.tolist()}')
print(f'\nTypes:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nRating distribution:\n{df["note"].value_counts().sort_index()}')
print(f'\nInsurers  : {df["assureur"].unique()}')
display(df.head(5))

---
## Cell 3 — Data cleaning

In [ ]:
df = df.dropna(subset=['avis_en']).reset_index(drop=True)

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['avis_clean'] = df['avis_en'].apply(clean_text)
df['tokens']     = df['avis_clean'].apply(word_tokenize)

print(f'Before : {df["avis_en"].iloc[0]}')
print(f'After  : {df["avis_clean"].iloc[0]}')
print(f'\nAverage length (words) : {df["tokens"].apply(len).mean():.1f}')
print(f'Median length          : {df["tokens"].apply(len).median():.0f}')

open('./checkpoints/_cell3.done', 'w').close()
print('Cell 3 saved.')

---
## Cell 4 — Word frequencies and n-grams

In [ ]:
all_words   = [w for tokens in df['tokens'] for w in tokens]
word_freq   = Counter(all_words)
top_words   = word_freq.most_common(30)

all_bigrams  = [' '.join(bg) for tokens in df['tokens'] for bg in ngrams(tokens, 2)]
all_trigrams = [' '.join(tg) for tokens in df['tokens'] for tg in ngrams(tokens, 3)]
top_bigrams  = Counter(all_bigrams).most_common(20)
top_trigrams = Counter(all_trigrams).most_common(15)

fig, axes = plt.subplots(2, 2, figsize=(20, 14))

words, counts = zip(*top_words)
axes[0,0].barh(words, counts, color='steelblue')
axes[0,0].set_title('Top 30 words')
axes[0,0].invert_yaxis()

bg_w, bg_c = zip(*top_bigrams)
axes[0,1].barh(bg_w, bg_c, color='salmon')
axes[0,1].set_title('Top 20 bigrams')
axes[0,1].invert_yaxis()

tg_w, tg_c = zip(*top_trigrams)
axes[1,0].barh(tg_w, tg_c, color='mediumseagreen')
axes[1,0].set_title('Top 15 trigrams')
axes[1,0].invert_yaxis()

wc = WordCloud(width=800, height=400, background_color='white', colormap='viridis', max_words=100)
wc.generate(' '.join(all_words))
axes[1,1].imshow(wc, interpolation='bilinear')
axes[1,1].axis('off')
axes[1,1].set_title('WordCloud')

plt.tight_layout()
plt.savefig('word_frequencies.png', dpi=150)
plt.show()
print(f'Top 5 bigrams  : {top_bigrams[:5]}')
print(f'Top 5 trigrams : {top_trigrams[:5]}')

open('./checkpoints/_cell4.done', 'w').close()
print('Cell 4 saved.')

---
## Cell 5 — Sentiment labels

In [ ]:
def note_to_sentiment(note):
    if note <= 2.0:   return 0
    elif note == 3.0: return 1
    else:             return 2

df['label']         = df['note'].apply(note_to_sentiment)
df['sentiment_str'] = df['label'].map({0: 'negative', 1: 'neutral', 2: 'positive'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['label'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0],
    color=['tomato','gold','mediumseagreen'], edgecolor='black'
)
axes[0].set_xticklabels(['Negative','Neutral','Positive'], rotation=0)
axes[0].set_title('Sentiment distribution')
df['note'].hist(bins=10, ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Raw rating distribution')
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150)
plt.show()
print(df['label'].value_counts().sort_index())

open('./checkpoints/_cell5.done', 'w').close()
print('Cell 5 saved.')

---
## Cell 6 — Train / val / test split

In [ ]:
train_val_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label']
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.176, random_state=42, stratify=train_val_df['label']
)

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

print(f'Train      : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)')
print(f'Validation : {len(val_df)}   ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test       : {len(test_df)}  ({len(test_df)/len(df)*100:.1f}%)')
for name, split in [('Train',train_df),('Val',val_df),('Test',test_df)]:
    d = split['label'].value_counts(normalize=True).sort_index()
    print(f'{name}: neg={d[0]:.2f} neu={d[1]:.2f} pos={d[2]:.2f}')

train_df.to_csv('./checkpoints/train_df.csv', index=False)
val_df.to_csv(  './checkpoints/val_df.csv',   index=False)
test_df.to_csv( './checkpoints/test_df.csv',  index=False)
np.save('./checkpoints/y_train.npy', y_train)
np.save('./checkpoints/y_val.npy',   y_val)
np.save('./checkpoints/y_test.npy',  y_test)
df.to_csv('./checkpoints/df_full.csv', index=False)
open('./checkpoints/_cell6.done', 'w').close()
print('Cell 6 saved.')

---
# ============================================================
# SECTION INDEPENDANTE — APRES SPLIT TRAIN/VAL/TEST (CELLS 7 A 10)
# POINT DE REPRISE APRES KERNEL RESTART
#
# CONDITIONS REQUISES :
#   - cells 1 a 6 executees ET checkpoints/train_df.csv present
#   - checkpoints/val_df.csv, test_df.csv, y_*.npy, df_full.csv presents
#
# VARIABLES RECHARGEES :
#   - df, train_df, val_df, test_df, y_train, y_val, y_test
#   - tokens reconstruit depuis avis_clean
# ============================================================

In [ ]:
# Rechargement minimal apres kernel restart — necessite cell 1 a 6 executees au moins une fois
import pandas as pd, numpy as np, pickle, os, warnings
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

df       = pd.read_csv('./checkpoints/df_full.csv')
train_df = pd.read_csv('./checkpoints/train_df.csv')
val_df   = pd.read_csv('./checkpoints/val_df.csv')
test_df  = pd.read_csv('./checkpoints/test_df.csv')
for _d in [df, train_df, val_df, test_df]:
    _d['avis_en']    = _d['avis_en'].fillna('')
    _d['avis_clean'] = _d['avis_clean'].fillna('')
y_train  = np.load('./checkpoints/y_train.npy')
y_val    = np.load('./checkpoints/y_val.npy')
y_test   = np.load('./checkpoints/y_test.npy')

# Reconstruire tokens depuis avis_clean
stop_words = set(stopwords.words('english'))
df['tokens'] = df['avis_clean'].fillna('').apply(word_tokenize)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'df: {len(df)} lignes | train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}')
print('Rechargement Point 1 OK.')

---
## Cell 7 — LDA topic modeling

In [ ]:
N_TOPICS = 8

# Garantir aucun NaN dans avis_clean quelle que soit l'origine du df
df['avis_clean'] = df['avis_clean'].fillna('').astype(str)
df['avis_en']    = df['avis_en'].fillna('').astype(str)

tfidf_lda = TfidfVectorizer(max_features=5000, min_df=3, max_df=0.85)
dtm       = tfidf_lda.fit_transform(df['avis_clean'])

lda = LatentDirichletAllocation(
    n_components=N_TOPICS, max_iter=30,
    learning_method='online', random_state=42, n_jobs=-1
)
lda.fit(dtm)

vocab = tfidf_lda.get_feature_names_out()
topic_labels = [
    'Price & Contract', 'Customer Service', 'Claims & Reimbursement',
    'Mobile App', 'Overall Quality', 'Responsiveness',
    'Cancellation', 'Recommendation'
]

topic_words = {}
for i, topic in enumerate(lda.components_):
    top_idx   = topic.argsort()[-15:][::-1]
    top_terms = [vocab[j] for j in top_idx]
    topic_words[i] = top_terms
    print(f'Topic {i} — {topic_labels[i]}: {top_terms[:8]}')

df['dominant_topic'] = lda.transform(dtm).argmax(axis=1)
df['topic_label']    = df['dominant_topic'].apply(lambda x: topic_labels[x])

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for i, ax in enumerate(axes.flatten()):
    if i < N_TOPICS:
        wc_t = WordCloud(width=300, height=200, background_color='white')
        freq_dict = {vocab[j]: lda.components_[i][j] for j in range(len(vocab))}
        wc_t.generate_from_frequencies(freq_dict)
        ax.imshow(wc_t, interpolation='bilinear')
        ax.set_title(topic_labels[i], fontsize=10)
        ax.axis('off')
plt.suptitle('LDA Topics', fontsize=14)
plt.tight_layout()
plt.savefig('lda_topics.png', dpi=150)
plt.show()
print(df['topic_label'].value_counts())

with open('./checkpoints/lda_model.pkl',            'wb') as f: pickle.dump(lda, f)
with open('./checkpoints/tfidf_lda_vectorizer.pkl', 'wb') as f: pickle.dump(tfidf_lda, f)
with open('./checkpoints/topic_labels.pkl',         'wb') as f: pickle.dump(topic_labels, f)
open('./checkpoints/_cell7.done', 'w').close()
print('Cell 7 saved.')

---
## Cell 8 — Word2Vec training

In [ ]:
sentences = df['tokens'].tolist()

w2v_model = Word2Vec(
    sentences=sentences, vector_size=300, window=5,
    min_count=3, workers=8, sg=1, epochs=20, seed=42
)
w2v_model.save('word2vec_reviews.model')
print(f'Word2Vec vocabulary size: {len(w2v_model.wv)} words')

for word in ['insurance', 'price', 'service', 'claim', 'cancel']:
    if word in w2v_model.wv:
        print(f'Closest to "{word}" : {w2v_model.wv.most_similar(word, topn=5)}')

open('./checkpoints/_cell8.done', 'w').close()
print('Cell 8 saved.')

---
# ============================================================
# SECTION INDEPENDANTE — APRES WORD2VEC (CELLS 9 A 13 ET 14)
# POINT DE REPRISE APRES KERNEL RESTART
#
# CONDITIONS REQUISES :
#   - cells 1 a 8 executees ET word2vec_reviews.model present
#   - checkpoints/train_df.csv, df_full.csv, y_*.npy presents
#
# VARIABLES RECHARGEES :
#   - df, train_df, val_df, test_df, y_*
#   - w2v_model
#   - device
# ============================================================

In [ ]:
# Rechargement minimal apres kernel restart — necessite cells 1 a 8 executees au moins une fois
import pandas as pd, numpy as np, pickle, os, warnings
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

df       = pd.read_csv('./checkpoints/df_full.csv')
train_df = pd.read_csv('./checkpoints/train_df.csv')
val_df   = pd.read_csv('./checkpoints/val_df.csv')
test_df  = pd.read_csv('./checkpoints/test_df.csv')
for _d in [df, train_df, val_df, test_df]:
    _d['avis_en']    = _d['avis_en'].fillna('')
    _d['avis_clean'] = _d['avis_clean'].fillna('')
y_train  = np.load('./checkpoints/y_train.npy')
y_val    = np.load('./checkpoints/y_val.npy')
y_test   = np.load('./checkpoints/y_test.npy')

df['tokens'] = df['avis_clean'].fillna('').apply(word_tokenize)

from gensim.models import Word2Vec
w2v_model = Word2Vec.load('word2vec_reviews.model')
print(f'Word2Vec: {len(w2v_model.wv)} mots')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print('Rechargement Point 2 OK.')

---
## Cell 9 — GloVe pre-trained embeddings

Modele : `glove-wiki-gigaword-50` (69 MB) — remplace `glove-wiki-gigaword-300` (1.6 GB)

In [ ]:
print('Loading GloVe glove-wiki-gigaword-50...')
glove_model = api.load('glove-wiki-gigaword-50')
print(f'GloVe loaded — {len(glove_model)} vectors dim 50')

for word in ['insurance', 'accident', 'refund', 'customer']:
    if word in glove_model:
        print(f'GloVe closest to "{word}" : {glove_model.most_similar(word, topn=5)}')

if all(w in glove_model for w in ['king','man','woman']):
    res = glove_model.most_similar(positive=['king','woman'], negative=['man'], topn=3)
    print(f'\nAnalogy king - man + woman = {res}')

open('./checkpoints/_cell9.done', 'w').close()
print('Cell 9 saved.')

---
## Cell 10 — Cosine and Euclidean distance

In [ ]:
def cosine_similarity(v1, v2): return 1 - cosine(v1, v2)
def euclidean_distance(v1, v2): return euclidean(v1, v2)

word_pairs = [
    ('insurance','policy'), ('insurance','accident'),
    ('good','excellent'),   ('good','terrible'),
    ('price','cost'),       ('price','happiness'),
]
print(f'{"Pair":<35} {"Cosine":>10} {"Euclidean":>12}')
print('-'*60)
for w1, w2 in word_pairs:
    if w1 in glove_model and w2 in glove_model:
        v1, v2 = glove_model[w1], glove_model[w2]
        print(f'{w1} vs {w2:<25} {cosine_similarity(v1,v2):>10.4f} {euclidean_distance(v1,v2):>12.4f}')

open('./checkpoints/_cell10.done', 'w').close()
print('Cell 10 saved.')

---
## Cell 11 — Semantic search with SBERT and FAISS

Modele : `all-MiniLM-L6-v2` (80 MB) — remplace `all-mpnet-base-v2` (420 MB)

In [ ]:
import faiss

print('Loading Sentence-BERT (all-MiniLM-L6-v2)...')
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))

print('Encoding reviews...')
review_texts = df['avis_en'].tolist()
embeddings   = sbert_model.encode(
    review_texts, batch_size=128,
    show_progress_bar=True, normalize_embeddings=True
)
np.save('review_embeddings.npy', embeddings)

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype('float32'))
faiss.write_index(index, 'faiss_reviews.index')
print(f'FAISS index: {index.ntotal} vectors, dim {dim}')

del sbert_model
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('SBERT unloaded from memory')

open('./checkpoints/_cell11.done', 'w').close()
print('Cell 11 saved.')


def semantic_search(query, k=5):
    _index = faiss.read_index('faiss_reviews.index')
    _m = SentenceTransformer('all-MiniLM-L6-v2')
    q_vec = _m.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = _index.search(q_vec, k)
    for score, idx in zip(scores[0], indices[0]):
        print(f'  [{score:.3f}] {df["note"].iloc[idx]} stars {df["assureur"].iloc[idx]} — {df["avis_en"].iloc[idx][:80]}...')

print('\nTest semantic search:')
semantic_search('terrible customer service very slow', k=3)

---
## Cell 12 — Word2Vec embeddings visualization with UMAP

In [ ]:
vocab_viz   = list(w2v_model.wv.key_to_index.keys())[:300]
vectors_viz = np.array([w2v_model.wv[w] for w in vocab_viz])

reducer      = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding_2d = reducer.fit_transform(vectors_viz)

plt.figure(figsize=(16, 12))
plt.scatter(embedding_2d[:,0], embedding_2d[:,1], alpha=0.5, s=15, c='steelblue')

highlight = ['insurance','price','service','claim','cancel','good','bad',
             'excellent','terrible','fast','slow','refund','accident','policy']
for word in highlight:
    if word in vocab_viz:
        idx = vocab_viz.index(word)
        plt.annotate(word, xy=(embedding_2d[idx,0], embedding_2d[idx,1]),
                     fontsize=11, color='crimson', fontweight='bold')
        plt.scatter(embedding_2d[idx,0], embedding_2d[idx,1], s=80, c='crimson', zorder=5)

plt.title('Word2Vec embeddings — UMAP 2D', fontsize=16)
plt.savefig('embeddings_umap.png', dpi=150)
plt.show()

open('./checkpoints/_cell12.done', 'w').close()
print('Cell 12 saved.')

---
## Cell 13 — TensorBoard embeddings export

In [ ]:
os.makedirs('tensorboard_logs/embeddings', exist_ok=True)
tb_writer = SummaryWriter('tensorboard_logs/embeddings')

viz_words   = list(w2v_model.wv.key_to_index.keys())[:500]
viz_vectors = torch.tensor(
    np.array([w2v_model.wv[w] for w in viz_words]), dtype=torch.float32
)
tb_writer.add_embedding(viz_vectors, metadata=viz_words, tag='Word2Vec_embeddings')
tb_writer.close()

print('TensorBoard embeddings saved')
print('  tensorboard --logdir=tensorboard_logs/embeddings --port=6006')

open('./checkpoints/_cell13.done', 'w').close()
print('Cell 13 saved.')

---
## Cell 14 — Model 1 : TF-IDF with classical classifiers

In [ ]:
df['avis_clean'] = df['avis_clean'].fillna('').astype(str)
train_df['avis_clean'] = train_df['avis_clean'].fillna('').astype(str)
val_df['avis_clean']   = val_df['avis_clean'].fillna('').astype(str)
test_df['avis_clean']  = test_df['avis_clean'].fillna('').astype(str)

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
X_train_tfidf = tfidf.fit_transform(train_df['avis_clean'])
X_val_tfidf   = tfidf.transform(val_df['avis_clean'])
X_test_tfidf  = tfidf.transform(test_df['avis_clean'])

tfidf_models = {
    'Logistic Regression': LogisticRegression(max_iter=500, C=1.0, random_state=42),
    'Linear SVM':          LinearSVC(max_iter=2000, C=1.0, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
}
tfidf_results = {}
for name, clf in tfidf_models.items():
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average='weighted')
    tfidf_results[name] = {'accuracy': acc, 'f1': f1, 'preds': preds}
    print(f'{name}: Acc={acc:.4f} | F1={f1:.4f}')

best_tfidf = max(tfidf_results, key=lambda k: tfidf_results[k]['f1'])
print(f'\nBest TF-IDF model: {best_tfidf}')
print(classification_report(y_test, tfidf_results[best_tfidf]['preds'],
                             target_names=['negative','neutral','positive']))

with open('./checkpoints/tfidf_vectorizer.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('./checkpoints/tfidf_results.pkl',    'wb') as f:
    pickle.dump({k: {'preds': v['preds']} for k, v in tfidf_results.items()}, f)
open('./checkpoints/_cell14.done', 'w').close()
print('Cell 14 saved.')

---
# ============================================================
# SECTION INDEPENDANTE — APRES TF-IDF CLASSIQUES (CELLS 15 A 16)
# POINT DE REPRISE APRES KERNEL RESTART
#
# CONDITIONS REQUISES :
#   - cells 1 a 14 executees ET checkpoints/tfidf_vectorizer.pkl present
#   - checkpoints/tfidf_results.pkl, train_df.csv, df_full.csv presents
#
# VARIABLES RECHARGEES :
#   - df, train_df, val_df, test_df, y_*
#   - tfidf, tfidf_results
#   - MAX_VOCAB=20000, MAX_LEN=128
# ============================================================

In [ ]:
# Rechargement minimal apres kernel restart — necessite cells 1 a 14 executees au moins une fois
import pandas as pd, numpy as np, pickle, os, warnings
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

df       = pd.read_csv('./checkpoints/df_full.csv')
train_df = pd.read_csv('./checkpoints/train_df.csv')
val_df   = pd.read_csv('./checkpoints/val_df.csv')
test_df  = pd.read_csv('./checkpoints/test_df.csv')
for _d in [df, train_df, val_df, test_df]:
    _d['avis_en']    = _d['avis_en'].fillna('')
    _d['avis_clean'] = _d['avis_clean'].fillna('')
y_train  = np.load('./checkpoints/y_train.npy')
y_val    = np.load('./checkpoints/y_val.npy')
y_test   = np.load('./checkpoints/y_test.npy')

with open('./checkpoints/tfidf_vectorizer.pkl', 'rb') as f: tfidf         = pickle.load(f)
with open('./checkpoints/tfidf_results.pkl',    'rb') as f: tfidf_results = pickle.load(f)

MAX_VOCAB = 20000
MAX_LEN   = 128

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
print(f'Device : {device}')
print('Rechargement Point 3 OK.')

---
## Cell 15 — Model 2 : CNN with learned embedding layer

In [ ]:
import time
MAX_VOCAB  = 20000
MAX_LEN    = 128
EMBED_DIM  = 128
BATCH_SIZE = 256

t0 = time.time()
print('Fitting Keras tokenizer...')
keras_tok = KerasTokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
keras_tok.fit_on_texts(train_df['avis_clean'])
print(f'  Done in {time.time()-t0:.1f}s')

def encode_keras(texts):
    seqs = keras_tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

print('Encoding sequences...')
X_tr = encode_keras(train_df['avis_clean'])
X_vl = encode_keras(val_df['avis_clean'])
X_te = encode_keras(test_df['avis_clean'])
print(f'  Train {X_tr.shape} | Val {X_vl.shape} | Test {X_te.shape}')

emb_model = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN, name='embedding_layer'),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])
emb_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
emb_model.build(input_shape=(None, MAX_LEN))
emb_model.summary()

os.makedirs('tensorboard_logs/embedding_model', exist_ok=True)
tb_emb = TFTensorBoard(log_dir='tensorboard_logs/embedding_model', histogram_freq=1)

print('\nTraining CNN embedding model...')
t1 = time.time()
history_emb = emb_model.fit(
    X_tr, y_train, validation_data=(X_vl, y_val),
    epochs=10, batch_size=BATCH_SIZE, callbacks=[tb_emb], verbose=1
)

preds_emb = emb_model.predict(X_te, verbose=1).argmax(axis=1)
print(f'\nCNN Embedding — Acc: {accuracy_score(y_test, preds_emb):.4f} | F1: {f1_score(y_test, preds_emb, average="weighted"):.4f}')
print(f'Total time: {time.time()-t0:.1f}s')

emb_model.save('./checkpoints/emb_model.keras')
with open('./checkpoints/keras_tokenizer.pkl', 'wb') as f: pickle.dump(keras_tok, f)
np.save('./checkpoints/preds_emb.npy', preds_emb)
open('./checkpoints/_cell15.done', 'w').close()
print('Cell 15 saved.')

---
## Cell 16 — Model 3 : Bi-LSTM with GloVe embeddings

Matrice GloVe construite depuis `glove-wiki-gigaword-50` (dim 50).

In [ ]:
import time

t0 = time.time()
print('Building GloVe embedding matrix (dim 50)...')
glove_dim    = 50
embed_matrix = np.zeros((MAX_VOCAB, glove_dim))
n_found      = 0
vocab_size   = min(MAX_VOCAB, len(keras_tok.word_index))

for i, (word, idx) in enumerate(keras_tok.word_index.items()):
    if idx < MAX_VOCAB and word in glove_model:
        embed_matrix[idx] = glove_model[word]
        n_found += 1
    if (i+1) % 5000 == 0:
        print(f'  {i+1}/{vocab_size} words — {n_found} GloVe hits')
print(f'GloVe hits: {n_found}/{vocab_size} ({n_found/vocab_size*100:.1f}%) — {time.time()-t0:.1f}s')

del glove_model
import gc; gc.collect()
print('GloVe unloaded from memory')

lstm_model = Sequential([
    Embedding(MAX_VOCAB, glove_dim, weights=[embed_matrix],
              input_length=MAX_LEN, trainable=True, name='glove_embedding'),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3)),
    Bidirectional(LSTM(64, dropout=0.3)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model.build(input_shape=(None, MAX_LEN))
lstm_model.summary()

os.makedirs('tensorboard_logs/lstm_model', exist_ok=True)
tb_lstm = TFTensorBoard(log_dir='tensorboard_logs/lstm_model', histogram_freq=1)

print('\nTraining Bi-LSTM...')
t1 = time.time()
history_lstm = lstm_model.fit(
    X_tr, y_train, validation_data=(X_vl, y_val),
    epochs=12, batch_size=64, callbacks=[tb_lstm], verbose=1
)

preds_lstm = lstm_model.predict(X_te, verbose=1).argmax(axis=1)
print(f'\nBi-LSTM + GloVe — Acc: {accuracy_score(y_test, preds_lstm):.4f} | F1: {f1_score(y_test, preds_lstm, average="weighted"):.4f}')
print(f'Total time: {time.time()-t0:.1f}s')

lstm_model.save('./checkpoints/lstm_model.keras')
np.save('./checkpoints/preds_lstm.npy', preds_lstm)
open('./checkpoints/_cell16.done', 'w').close()
print('Cell 16 saved.')

In [ ]:
os.makedirs('./checkpoints', exist_ok=True)

train_df.to_csv('./checkpoints/train_df.csv', index=False)
val_df.to_csv(  './checkpoints/val_df.csv',   index=False)
test_df.to_csv( './checkpoints/test_df.csv',  index=False)

np.save('./checkpoints/y_train.npy', y_train)
np.save('./checkpoints/y_val.npy',   y_val)
np.save('./checkpoints/y_test.npy',  y_test)

with open('./checkpoints/tfidf_vectorizer.pkl',    'wb') as f: pickle.dump(tfidf, f)
with open('./checkpoints/tfidf_results.pkl',       'wb') as f:
    pickle.dump({k: {'preds': v['preds']} for k, v in tfidf_results.items()}, f)
with open('./checkpoints/keras_tokenizer.pkl',     'wb') as f: pickle.dump(keras_tok, f)
with open('./checkpoints/lda_model.pkl',           'wb') as f: pickle.dump(lda, f)
with open('./checkpoints/tfidf_lda_vectorizer.pkl','wb') as f: pickle.dump(tfidf_lda, f)
with open('./checkpoints/topic_labels.pkl',        'wb') as f: pickle.dump(topic_labels, f)

np.save('./checkpoints/preds_emb.npy',     preds_emb)
np.save('./checkpoints/preds_lstm.npy',    preds_lstm)
np.save('./checkpoints/preds_deberta.npy', preds_deberta)
np.save('./checkpoints/preds_mistral.npy', preds_mistral)

emb_model.save( './checkpoints/emb_model.keras')
lstm_model.save('./checkpoints/lstm_model.keras')

df.to_csv('./checkpoints/df_full.csv', index=False)

print('Checkpoint complete')
print(f'  preds_deberta : {preds_deberta.shape}')
print(f'  preds_mistral : {preds_mistral.shape}')

---
# ============================================================
# SECTION INDEPENDANTE — COMPARAISON FINALE
# POINT DE REPRISE APRES KERNEL RESTART
#
# CONDITIONS : notebooks 1 ET 2 complets,
# tous les preds_*.npy presents dans ./checkpoints/
# ============================================================

In [ ]:
# Rechargement pour la comparaison finale — necessite notebooks 1 et 2 complets
import pandas as pd, numpy as np, pickle, os, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
warnings.filterwarnings('ignore')

df      = pd.read_csv('./checkpoints/df_full.csv')
test_df = pd.read_csv('./checkpoints/test_df.csv')
for _d in [df, test_df]:
    _d['avis_en']    = _d['avis_en'].fillna('').astype(str)
    _d['avis_clean'] = _d['avis_clean'].fillna('').astype(str)

y_test        = np.load('./checkpoints/y_test.npy')
preds_emb     = np.load('./checkpoints/preds_emb.npy')
preds_lstm    = np.load('./checkpoints/preds_lstm.npy')
preds_deberta = np.load('./checkpoints/preds_deberta.npy')
preds_mistral = np.load('./checkpoints/preds_mistral.npy')

with open('./checkpoints/tfidf_results.pkl',   'rb') as f: tfidf_results = pickle.load(f)
with open('./checkpoints/lda_model.pkl',       'rb') as f: lda           = pickle.load(f)
with open('./checkpoints/topic_labels.pkl',    'rb') as f: topic_labels  = pickle.load(f)

print('Rechargement comparaison OK.')

---
## Cell 19 — Model comparison

In [ ]:
all_preds = {
    'TF-IDF + LR':           tfidf_results['Logistic Regression']['preds'],
    'TF-IDF + SVM':          tfidf_results['Linear SVM']['preds'],
    'TF-IDF + RF':           tfidf_results['Random Forest']['preds'],
    'CNN Emb. Layer':        preds_emb,
    'Bi-LSTM + GloVe-50':    preds_lstm,
    'DeBERTa-v3-small':      preds_deberta,
    'Mistral-7B QLoRA':      preds_mistral,
}

comparison = []
for name, preds in all_preds.items():
    comparison.append({
        'Model':         name,
        'Accuracy':      round(accuracy_score(y_test, preds), 4),
        'F1 (weighted)': round(f1_score(y_test, preds, average='weighted'), 4),
        'F1 (macro)':    round(f1_score(y_test, preds, average='macro'), 4),
    })

comp_df = pd.DataFrame(comparison).sort_values('F1 (weighted)', ascending=False)
print(comp_df.to_string(index=False))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(comp_df)))
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

bars = axes[0].barh(comp_df['Model'], comp_df['Accuracy'], color=colors[::-1])
axes[0].set_xlim(0, 1)
axes[0].set_title('Accuracy by model', fontsize=14)
for bar, val in zip(bars, comp_df['Accuracy']):
    axes[0].text(val+0.005, bar.get_y()+bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10)

bars2 = axes[1].barh(comp_df['Model'], comp_df['F1 (weighted)'], color=colors[::-1])
axes[1].set_xlim(0, 1)
axes[1].set_title('Weighted F1-score by model', fontsize=14)
for bar, val in zip(bars2, comp_df['F1 (weighted)']):
    axes[1].text(val+0.005, bar.get_y()+bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()
comp_df.to_csv('model_comparison.csv', index=False)

---
## Cell 20 — Error analysis

In [ ]:
best_preds  = preds_deberta
label_names = ['negative','neutral','positive']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cm     = confusion_matrix(y_test, best_preds)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:,np.newaxis]

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=label_names, yticklabels=label_names)
axes[0].set_title('Confusion matrix (DeBERTa-v3-small)')
axes[0].set_ylabel('True label'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_pct, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=label_names, yticklabels=label_names)
axes[1].set_title('Normalized confusion matrix (%)')
axes[1].set_ylabel('True label'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_test, best_preds, target_names=label_names))

test_copy            = test_df.copy().reset_index(drop=True)
test_copy['pred']    = best_preds
test_copy['correct'] = (test_copy['label'] == test_copy['pred'])
errors = test_copy[~test_copy['correct']]

print(f'Errors: {len(errors)}/{len(test_copy)} ({len(errors)/len(test_copy)*100:.1f}%)')
print(f'\nErrors by true label:')
print(errors['label'].value_counts().sort_index())

print('\n5 error examples:')
for _, row in errors.sample(min(5,len(errors)), random_state=42).iterrows():
    print(f'  True: {label_names[int(row["label"])]} | Predicted: {label_names[int(row["pred"])]}')
    print(f'  Review: {str(row["avis_en"])[:100]}...')

---
## Cell 21 — Analysis by insurer and topic

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

avg_note = df.groupby('assureur')['note'].agg(['mean','count']).round(2).sort_values('mean', ascending=False)
bars = axes[0,0].bar(avg_note.index, avg_note['mean'], color='steelblue', edgecolor='black')
axes[0,0].set_ylim(0, 5.5)
axes[0,0].set_title('Average rating by insurer')
axes[0,0].tick_params(axis='x', rotation=45)
for bar, (_, row) in zip(bars, avg_note.iterrows()):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                   f'{row["mean"]:.2f}\n(n={int(row["count"])})', ha='center', fontsize=9)

sent_by_ass = df.groupby(['assureur','sentiment_str']).size().unstack(fill_value=0)
sent_pct    = sent_by_ass.div(sent_by_ass.sum(axis=1), axis=0)
sent_pct.plot(kind='bar', ax=axes[0,1], stacked=True,
              color=['tomato','gold','mediumseagreen'])
axes[0,1].set_title('Sentiment breakdown by insurer')
axes[0,1].tick_params(axis='x', rotation=45)

topic_sent = df.groupby(['topic_label','sentiment_str']).size().unstack(fill_value=0)
topic_pct  = topic_sent.div(topic_sent.sum(axis=1), axis=0)
topic_pct.plot(kind='barh', ax=axes[1,0], stacked=True,
               color=['tomato','gold','mediumseagreen'])
axes[1,0].set_title('Sentiment by topic')

pivot = df.groupby(['assureur','topic_label'])['note'].mean().unstack(fill_value=np.nan)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=1, vmax=5, ax=axes[1,1])
axes[1,1].set_title('Average rating: insurer x topic')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('insurer_analysis.png', dpi=150)
plt.show()

---
## Cell 22 — Save assets for Streamlit

In [ ]:
df.to_csv('reviews_with_topics_sentiment.csv', index=False)

with open('tfidf_vectorizer.pkl',     'wb') as f: pickle.dump(tfidf, f)
with open('logreg_model.pkl',         'wb') as f: pickle.dump(tfidf_models['Logistic Regression'], f)
with open('keras_tokenizer.pkl',      'wb') as f: pickle.dump(keras_tok, f)
with open('topic_labels.pkl',         'wb') as f: pickle.dump(topic_labels, f)
with open('lda_model.pkl',            'wb') as f: pickle.dump(lda, f)
with open('tfidf_lda_vectorizer.pkl', 'wb') as f: pickle.dump(tfidf_lda, f)

df[['avis_en','assureur','produit','note','sentiment_str','topic_label']].to_csv(
    'reviews_for_streamlit.csv', index=False
)
comp_df.to_csv('model_comparison.csv', index=False)

print('Streamlit assets saved:')
print('  reviews_for_streamlit.csv, tfidf_vectorizer.pkl,')
print('  logreg_model.pkl, keras_tokenizer.pkl')
print('  review_embeddings.npy, faiss_reviews.index')
print('  best_deberta_model/, best_mistral_lora/')

---
## Cell 23 — Launch Streamlit

In [ ]:
print('=== Run Streamlit ===')
print('In the SSH terminal:')
print('  source ~/ESILV_NPL_PROJECT/venv_nlp/bin/activate')
print('  streamlit run streamlit_app.py --server.port 8501')
print()
print('From your local machine:')
print('  ssh -L 8501:localhost:8501 user@your_server')
print('  Then open: http://localhost:8501')
print()
print('TensorBoard (optional):')
print('  tensorboard --logdir=tensorboard_logs --port=6006')
print('  ssh -L 6006:localhost:6006 user@your_server')